## **Dự đoán tình trạng máy móc thiết bị công nghiệp bị khi đưa vào vận hành**.   

### **Data processing** 

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

# Hỗ trợ chạy notebook từ repo root hoặc trực tiếp trong thư mục classification.
working_dir = Path.cwd().resolve()
repo_candidates = (working_dir, *working_dir.parents)
REPO_ROOT = next((
    path for path in repo_candidates
    if (path / "classification" / "lightgbm_classification.py").is_file()
), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy repo root chứa classification/lightgbm_classification.py."
    )

repo_root_text = str(REPO_ROOT)
if repo_root_text not in sys.path:
    sys.path.insert(0, repo_root_text)

from classification.evaluation.adapter import (
    evaluate_classification_outputs,
)
from classification.lightgbm_classification import LightGBMClassification
from classification.evaluation.run_machine_failure_evaluation import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)


In [ ]:
# Đọc dữ liệu 
DATA_PATH = REPO_ROOT / "classification" / "data" / "raw" / "machine_fail.csv"
df = pd.read_csv(DATA_PATH)
print ('Kích thước dataset', df.shape ) 

df.head ( 10 )

In [ ]:
# Kiểm tra kiểu dữ liệu từng feature  
df.dtypes  

In [ ]:
# Dữ liệu thiếu 
df.isnull ( ).sum ( )

In [ ]:
# Chuẩn hóa dạng số của type of machine 
print ( df['Type'].value_counts ( ))   

df["Type"] = df["Type"].map({
    "L": 0,
    "M": 1,
    "H": 2
}) 

# Sau chuẩn hóa  
df.head ( )

In [ ]:
# Chia dữ liệu thành 2 tập train và test 
X_features = df.loc[:, FEATURE_COLUMNS].copy()

# Nhãn cần dự đoán
y_label = df[TARGET_COLUMN] 

# Chia 80% train - 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_label,
    test_size=0.2,
    random_state=42,
    stratify=y_label
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### **Training model** 

In [ ]:
# Tiến hành gọi thuật toán xây dựng và training model 
model_lightGBM_cls = LightGBMClassification(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=15,
    max_depth=5,
    random_state=42
) 

print (LightGBMClassification)

In [ ]:
# Training model  
model_lightGBM_cls.fit ( X_train , y_train )

In [ ]:
# Predict trên tập test  
predict_test =  model_lightGBM_cls.predict (X_test)   
predict_test_proba = model_lightGBM_cls.predict_proba ( X_test ) 

# So sánh đánh giá tổng quan so với giá trị thực  
print ('Kết quả dự đoán với 10 thiết bị đầu tiên :', predict_test[:10] )  
print ('Kết quả thực tế của 10 thiết bị đầu tiên :' )  
print (y_test[:10])
print ('=== Kết quả dự đoán dưới dạng xác xuất ====')
print (predict_test_proba[:10])

###   **Model Performance Evaluation** 

In [ ]:
# Tính metric thủ công, xuất CSV và tạo toàn bộ biểu đồ đánh giá.
EVALUATION_OUTPUT_DIR = REPO_ROOT / "classification" / "evaluation" / "outputs"
evaluation_result = evaluate_classification_outputs(
    model=model_lightGBM_cls,
    y_true=y_test.to_numpy(),
    y_pred=predict_test,
    y_proba=predict_test_proba,
    output_dir=EVALUATION_OUTPUT_DIR,
)

print(f"Đã lưu kết quả đánh giá tại: {EVALUATION_OUTPUT_DIR}")
evaluation_result["classification_report"]